# Pipeline treningu modelu (DigitalOcean Spaces)

Notebook realizuje pełny proces:
1. Wczytanie danych z **DigitalOcean Spaces** na podstawie `.env`.
2. Czyszczenie i przygotowanie danych.
3. Trening modelu regresyjnego w **PyCaret** + **feature selection**.
4. Ewaluacja metrykami: **MAE, RMSE, R2, MAPE**.
5. Zapis nowej wersji modelu lokalnie i upload do Spaces.

In [ ]:
# Jeśli trzeba, odkomentuj i uruchom:
# !pip install -q pandas pycaret boto3 python-dotenv

In [1]:
import io
import os
import re
import json
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse, urlunparse

import boto3
import numpy as np
import pandas as pd
from botocore.config import Config
from botocore.exceptions import BotoCoreError, ClientError
from dotenv import load_dotenv
from pycaret.regression import (
    compare_models,
    finalize_model,
    get_config,
    predict_model,
    pull,
    save_model,
    setup,
    tune_model,
)

In [2]:
load_dotenv()

SPACES_ENDPOINT = os.getenv("DO_SPACES_ENDPOINT")
SPACES_REGION = os.getenv("DO_SPACES_REGION", "fra1")
SPACES_BUCKET = os.getenv("DO_SPACES_BUCKET")
SPACES_ACCESS_KEY = os.getenv("DO_SPACES_ACCESS_KEY")
SPACES_SECRET_KEY = os.getenv("DO_SPACES_SECRET_KEY")

DATA_PREFIX = os.getenv("DO_SPACES_DATA_PREFIX", "data/")
MODEL_PREFIX = os.getenv("DO_SPACES_MODEL_PREFIX", "models/")
DATA_FILES = [
    x.strip() for x in os.getenv("DO_SPACES_DATA_FILES", "").split(",") if x.strip()
]

spaces_configured = all([SPACES_ENDPOINT, SPACES_BUCKET, SPACES_ACCESS_KEY, SPACES_SECRET_KEY])
if not spaces_configured:
    raise ValueError("Brakuje konfiguracji DO Spaces w .env. Pobieranie CSV działa wyłącznie z DigitalOcean Spaces.")
if str(SPACES_ACCESS_KEY).startswith('dop_v1'):
    raise ValueError("DO_SPACES_ACCESS_KEY wygląda jak token API DigitalOcean (dop_v1...). Użyj Access Key z Spaces (S3), nie API tokenu.")

LOCAL_MODELS_DIR = Path("models")
LOCAL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Konfiguracja załadowana.")
print("Tryb danych: tylko DigitalOcean Spaces")

Konfiguracja załadowana.
Tryb danych: tylko DigitalOcean Spaces


In [3]:
def normalize_spaces_endpoint(endpoint: str, bucket: str):
    parsed = urlparse(endpoint)
    host = parsed.netloc
    bucket_prefix = f"{bucket}."
    if host.startswith(bucket_prefix):
        host = host[len(bucket_prefix):]
    return urlunparse((parsed.scheme, host, parsed.path, parsed.params, parsed.query, parsed.fragment))


def make_spaces_client():
    endpoint = normalize_spaces_endpoint(SPACES_ENDPOINT, SPACES_BUCKET)
    return boto3.client(
        "s3",
        region_name=SPACES_REGION,
        endpoint_url=endpoint,
        aws_access_key_id=SPACES_ACCESS_KEY,
        aws_secret_access_key=SPACES_SECRET_KEY,
        config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
    )


def list_csv_keys(client, bucket: str, prefix: str):
    paginator = client.get_paginator("list_objects_v2")
    keys = []
    try:
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                if key.lower().endswith(".csv"):
                    keys.append(key)
    except (ClientError, BotoCoreError) as e:
        code = getattr(e, "response", {}).get("Error", {}).get("Code") if hasattr(e, "response") else None
        if code == "InvalidAccessKeyId":
            raise ValueError("DO_SPACES_ACCESS_KEY jest niepoprawny (InvalidAccessKeyId).") from e
        raise ValueError(f"Nie udało się pobrać listy plików z Spaces: {e}") from e
    return keys


def extract_year_from_key(key: str, default_year: int = 2024):
    match = re.search(r"(20\d{2})", key)
    return int(match.group(1)) if match else default_year


def read_csv_from_spaces(client, bucket: str, key: str):
    obj = client.get_object(Bucket=bucket, Key=key)
    body = obj["Body"].read()
    df = pd.read_csv(io.BytesIO(body), sep=";")
    df["source_key"] = key
    df["rok_biegu"] = extract_year_from_key(key)
    return df

In [4]:
spaces = make_spaces_client()

if not DATA_FILES:
    DATA_FILES = list_csv_keys(spaces, SPACES_BUCKET, DATA_PREFIX)

if not DATA_FILES:
    raise ValueError("Nie znaleziono plików CSV w Spaces. Ustaw DO_SPACES_DATA_FILES lub popraw DO_SPACES_DATA_PREFIX.")

frames = [read_csv_from_spaces(spaces, SPACES_BUCKET, key) for key in DATA_FILES]
print(f"Wczytano z Spaces plików: {len(frames)}")

raw_df = pd.concat(frames, ignore_index=True)
print(f"Rozmiar danych surowych: {raw_df.shape}")
raw_df.head()

Wczytano z Spaces plików: 2
Rozmiar danych surowych: (21957, 29)


,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo,source_key,rok_biegu
0,1.0,1787,TOMASZ,GRYCKO,NaN,POL,UKS BLIZA WŁADYSŁAWOWO,M,1.0,M30,...,1.0,3.106667,01:01:43,1.0,3.386667,0.031400,01:04:59,3.080509,data/halfmarathon_wroclaw_2023__final.csv,2023
1,2.0,3,ARKADIUSZ,GARDZIELEWSKI,WROCŁAW,POL,ARKADIUSZGARDZIELEWSKI.PL,M,2.0,M30,...,2.0,3.143333,01:03:08,2.0,3.540000,0.038000,01:06:23,3.146875,data/halfmarathon_wroclaw_2023__final.csv,2023
2,3.0,3832,KRZYSZTOF,HADAS,POZNAŃ,POL,NaN,M,3.0,M20,...,3.0,3.236667,01:05:09,3.0,3.516667,0.024067,01:08:24,3.242475,data/halfmarathon_wroclaw_2023__final.csv,2023
3,4.0,416,DAMIAN,DYDUCH,KĘPNO,POL,AZS POLITECHNIKA OPOLSKA,M,4.0,M30,...,5.0,3.330000,01:06:54,4.0,3.616667,0.025467,01:10:16,3.330963,data/halfmarathon_wroclaw_2023__final.csv,2023
4,5.0,8476,KAMIL,MAŃKOWSKI,MIRKÓW,POL,PARKRUN WROCŁAW,M,5.0,M20,...,7.0,3.386667,01:07:27,5.0,3.586667,0.023000,01:10:27,3.339654,data/halfmarathon_wroclaw_2023__final.csv,2023


In [5]:
def time_to_seconds(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        v = val.strip().upper()
        if v in {"", "DNS", "DNF", "DSQ"}:
            return np.nan
        parts = v.split(":")
        if len(parts) == 3:
            h, m, s = parts
            try:
                return int(h) * 3600 + int(m) * 60 + int(s)
            except ValueError:
                return np.nan
    return np.nan


def clean_data(df: pd.DataFrame):
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]

    for col in ["5 km Czas", "10 km Czas", "15 km Czas", "20 km Czas", "Czas"]:
        if col in out.columns:
            out[col] = out[col].apply(time_to_seconds)

    for col in ["5 km Tempo", "10 km Tempo", "15 km Tempo", "20 km Tempo", "Tempo Stabilność", "Rocznik"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "Rocznik" in out.columns and "rok_biegu" in out.columns:
        out["Wiek"] = out["rok_biegu"] - out["Rocznik"]

    if "Płeć" in out.columns:
        out["Płeć"] = out["Płeć"].astype(str).str.strip().str.upper()

    out = out.dropna(subset=["Czas"])
    out = out[out["Czas"] > 0]
    return out


clean_df = clean_data(raw_df)
print(f"Rozmiar danych po czyszczeniu: {clean_df.shape}")
clean_df.head()

Rozmiar danych po czyszczeniu: (18450, 30)


,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo,source_key,rok_biegu,Wiek
0,1.0,1787,TOMASZ,GRYCKO,NaN,POL,UKS BLIZA WŁADYSŁAWOWO,M,1.0,M30,...,3.106667,3703.0,1.0,3.386667,0.031400,3899.0,3.080509,data/halfmarathon_wroclaw_2023__final.csv,2023,31.0
1,2.0,3,ARKADIUSZ,GARDZIELEWSKI,WROCŁAW,POL,ARKADIUSZGARDZIELEWSKI.PL,M,2.0,M30,...,3.143333,3788.0,2.0,3.540000,0.038000,3983.0,3.146875,data/halfmarathon_wroclaw_2023__final.csv,2023,37.0
2,3.0,3832,KRZYSZTOF,HADAS,POZNAŃ,POL,NaN,M,3.0,M20,...,3.236667,3909.0,3.0,3.516667,0.024067,4104.0,3.242475,data/halfmarathon_wroclaw_2023__final.csv,2023,27.0
3,4.0,416,DAMIAN,DYDUCH,KĘPNO,POL,AZS POLITECHNIKA OPOLSKA,M,4.0,M30,...,3.330000,4014.0,4.0,3.616667,0.025467,4216.0,3.330963,data/halfmarathon_wroclaw_2023__final.csv,2023,35.0
4,5.0,8476,KAMIL,MAŃKOWSKI,MIRKÓW,POL,PARKRUN WROCŁAW,M,5.0,M20,...,3.386667,4047.0,5.0,3.586667,0.023000,4227.0,3.339654,data/halfmarathon_wroclaw_2023__final.csv,2023,28.0


In [6]:
# --- Wybór cech wejściowych ---
candidate_numeric = [
    "Wiek",
    "Rocznik",
    "5 km Czas",
    "10 km Czas",
    "15 km Czas",
    "20 km Czas",
    "5 km Tempo",
    "10 km Tempo",
    "15 km Tempo",
    "20 km Tempo",
    "Tempo Stabilność",
]
candidate_numeric = [c for c in candidate_numeric if c in clean_df.columns]

candidate_categorical = [c for c in ["Płeć", "Kategoria wiekowa"] if c in clean_df.columns]

target_col = "Czas"

selected_features = candidate_numeric + candidate_categorical

print("Cechy numeryczne:", candidate_numeric)
print("Cechy kategoryczne:", candidate_categorical)
print("Łącznie cech wejściowych:", len(selected_features))

Cechy numeryczne: ['Wiek', 'Rocznik', '5 km Czas', '10 km Czas', '15 km Czas', '20 km Czas', '5 km Tempo', '10 km Tempo', '15 km Tempo', '20 km Tempo', 'Tempo Stabilność']
Cechy kategoryczne: ['Płeć', 'Kategoria wiekowa']
Łącznie cech wejściowych: 13


In [7]:
model_df = clean_df[selected_features + [target_col]].copy()
model_df = model_df.dropna(subset=[target_col])

numeric_features = [c for c in selected_features if c in candidate_numeric]
categorical_features = [c for c in selected_features if c in candidate_categorical]

_ = setup(
    data=model_df,
    target=target_col,
    train_size=0.8,
    data_split_shuffle=True,
    data_split_stratify=False,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    session_id=42,
    fold=5,
    preprocess=True,
    normalize=True,
    feature_selection=True,
    feature_selection_method='classic',
    feature_selection_estimator='rf',
    verbose=False,
)

best_model = compare_models(sort='MAE')
leaderboard_top = pull().copy()
tuned_model = tune_model(best_model, optimize='MAE')
final_model = finalize_model(tuned_model)

pred_df = predict_model(final_model)
pred_col = 'prediction_label' if 'prediction_label' in pred_df.columns else 'Label'
y_test = pred_df[target_col].astype(float).to_numpy()
pred = pred_df[pred_col].astype(float).to_numpy()

abs_err = np.abs(y_test - pred)
sq_err = (y_test - pred) ** 2
den = np.clip(np.abs(y_test), 1e-9, None)
ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
r2 = 1.0 - (np.sum(sq_err) / ss_tot) if ss_tot > 0 else 0.0

metrics = {
    "MAE_seconds": float(np.mean(abs_err)),
    "RMSE_seconds": float(np.sqrt(np.mean(sq_err))),
    "R2": float(r2),
    "MAPE": float(np.mean(abs_err / den)),
}

selected_features_after_pycaret = get_config('X_train').columns.tolist()

print("Top modele (CV):")
display(leaderboard_top.head(5))
print("Liczba cech po feature selection (PyCaret):", len(selected_features_after_pycaret))
metrics

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
huber,Huber Regressor,39.2967,7722.8280,84.8179,0.9948,0.0103,0.0052,2.3800
par,Passive Aggressive Regressor,39.4182,7770.7337,85.1325,0.9947,0.0103,0.0052,2.3140
ridge,Ridge Regression,39.4741,7718.1877,84.8286,0.9948,0.0103,0.0052,2.8420
llar,Lasso Least Angle Regression,39.5188,7719.1639,84.8428,0.9948,0.0103,0.0052,2.8800
br,Bayesian Ridge,39.5193,7718.1376,84.8330,0.9948,0.0103,0.0052,2.3020
lr,Linear Regression,39.5198,7718.1437,84.8330,0.9948,0.0103,0.0052,3.0100
lar,Least Angle Regression,39.5198,7718.1437,84.8330,0.9948,0.0103,0.0052,2.9040
lasso,Lasso Regression,39.5760,7719.6544,84.8517,0.9948,0.0103,0.0052,2.8300
omp,Orthogonal Matching Pursuit,41.1097,7827.0340,85.6053,0.9947,0.0104,0.0055,2.2980
gbr,Gradient Boosting Regressor,41.8241,9200.5710,94.5068,0.9938,0.0116,0.0056,2.4240


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,38.6060,4703.4888,68.5820,0.9968,0.0085,0.0052
1,39.8114,14516.6589,120.4851,0.9903,0.0145,0.0052
2,39.1689,6533.6148,80.8308,0.9956,0.0098,0.0052
3,40.6207,9860.0969,99.2980,0.9931,0.0113,0.0053
4,38.1440,3034.8591,55.0896,0.9980,0.0071,0.0051
Mean,39.2702,7729.7437,84.8571,0.9948,0.0103,0.0052
Std,0.8760,4079.9879,23.0004,0.0027,0.0025,0.0001


Fitting 5 folds for each of 10 candidates, totalling 50 fits


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Huber Regressor,39.4496,10061.6939,100.3080,0.9933,0.0117,0.0052


Top modele (CV):


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
huber,Huber Regressor,39.2967,7722.8280,84.8179,0.9948,0.0103,0.0052,2.380
par,Passive Aggressive Regressor,39.4182,7770.7337,85.1325,0.9947,0.0103,0.0052,2.314
ridge,Ridge Regression,39.4741,7718.1877,84.8286,0.9948,0.0103,0.0052,2.842
llar,Lasso Least Angle Regression,39.5188,7719.1639,84.8428,0.9948,0.0103,0.0052,2.880
br,Bayesian Ridge,39.5193,7718.1376,84.8330,0.9948,0.0103,0.0052,2.302


Liczba cech po feature selection (PyCaret): 13


{'MAE_seconds': 39.4496308084728,
 'RMSE_seconds': 100.30799504061467,
 'R2': 0.9932656715258512,
 'MAPE': 0.00518803069676695}

In [ ]:
# --- Wersjonowanie + zapis lokalny ---
version = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
artifact_base = f"halfmarathon_time_model_{version}"
metrics_name = f"halfmarathon_time_model_{version}_metrics.json"

save_result = save_model(final_model, str(LOCAL_MODELS_DIR / artifact_base), verbose=False)
if isinstance(save_result, tuple):
    # W cz??ci wersji PyCaret save_model zwraca krotk?, np. (pipeline, path).
    model_path_value = next((x for x in save_result if isinstance(x, (str, os.PathLike))), None)
    if model_path_value is None:
        raise TypeError(f"Nieoczekiwany wynik save_model: {type(save_result)} -> {save_result}")
    local_model_path = Path(model_path_value)
else:
    local_model_path = Path(save_result)

local_metrics_path = LOCAL_MODELS_DIR / metrics_name

with open(local_metrics_path, "w", encoding="utf-8") as f:
    json.dump({
        "version": version,
        "metrics": metrics,
        "selected_features_before_pycaret": selected_features,
        "selected_features_after_pycaret": selected_features_after_pycaret,
        "target": target_col,
        "trained_at_utc": datetime.utcnow().isoformat(),
    }, f, ensure_ascii=False, indent=2)

# --- Budowa lekkiego modelu do inferencji w appce (mini) ---
# App używa 2 cech wyliczanych z opisu użytkownika: 20 km Czas i 15 km Tempo.
mini_features = ["20 km Czas", "15 km Tempo"]
missing_mini_features = [c for c in mini_features if c not in model_df.columns]
if missing_mini_features:
    raise ValueError(f"Brakuje cech do modelu mini: {missing_mini_features}")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import HuberRegressor
import joblib

model_input_cols = [c for c in model_df.columns if c != target_col]
mask = model_df[mini_features].notna().all(axis=1)
for c in model_input_cols:
    mask &= model_df[c].notna()

if not mask.any():
    raise ValueError("Brak wierszy treningowych do zbudowania modelu mini.")

mini_train_df = model_df.loc[mask, model_input_cols].copy()
X_mini = mini_train_df[mini_features].astype(float)
y_teacher = final_model.predict(mini_train_df).astype(float)

mini_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", HuberRegressor()),
])
mini_pipeline.fit(X_mini, y_teacher)

mini_mae_sec = float(np.mean(np.abs(mini_pipeline.predict(X_mini) - y_teacher)))

mini_name_versioned = f"halfmarathon_mini_{version}.pkl"
mini_name_latest = "halfmarathon_mini.pkl"
local_mini_versioned_path = LOCAL_MODELS_DIR / mini_name_versioned
local_mini_latest_path = LOCAL_MODELS_DIR / mini_name_latest

joblib.dump(mini_pipeline, str(local_mini_versioned_path), compress=3)
joblib.dump(mini_pipeline, str(local_mini_latest_path), compress=3)

print(f"Zapisano model: {local_model_path}")
print(f"Zapisano metryki: {local_metrics_path}")
print(f"Zapisano model mini (wersja): {local_mini_versioned_path}")
print(f"Zapisano model mini (latest): {local_mini_latest_path}")
print(f"Mini MAE vs teacher na zbiorze treningowym: {mini_mae_sec:.3f} s")

Zapisano model: models\halfmarathon_time_model_20260714_213310.pkl
Zapisano metryki: models\halfmarathon_time_model_20260714_213310_metrics.json
Zapisano model mini (wersja): models\halfmarathon_mini_20260714_213310.pkl
Zapisano model mini (latest): models\halfmarathon_mini.pkl
Mini MAE vs teacher na zbiorze treningowym: 0.000 s


In [ ]:
# --- Upload nowej wersji modelu do DigitalOcean Spaces ---
model_key = f"{MODEL_PREFIX.rstrip('/')}/{local_model_path.name}"
metrics_key = f"{MODEL_PREFIX.rstrip('/')}/{metrics_name}"
mini_key_versioned = f"{MODEL_PREFIX.rstrip('/')}/{local_mini_versioned_path.name}"
mini_key_latest = f"{MODEL_PREFIX.rstrip('/')}/{local_mini_latest_path.name}"

try:
    spaces.upload_file(str(local_model_path), SPACES_BUCKET, model_key)
    spaces.upload_file(str(local_metrics_path), SPACES_BUCKET, metrics_key)
    spaces.upload_file(str(local_mini_versioned_path), SPACES_BUCKET, mini_key_versioned)
    spaces.upload_file(str(local_mini_latest_path), SPACES_BUCKET, mini_key_latest)
    print("Upload zako?czony.")
    print(f"Model key:        {model_key}")
    print(f"Metrics key:      {metrics_key}")
    print(f"Mini key version: {mini_key_versioned}")
    print(f"Mini key latest:  {mini_key_latest}")
except (ClientError, BotoCoreError) as e:
    raise ValueError(f"Nie udało się wysłać artefaktów do Spaces: {e}") from e

Upload zako?czony.
Model key:        models/halfmarathon_time_model_20260714_213310.pkl
Metrics key:      models/halfmarathon_time_model_20260714_213310_metrics.json
Mini key version: models/halfmarathon_mini_20260714_213310.pkl
Mini key latest:  models/halfmarathon_mini.pkl
